# Stable Audio 3 — audio-conditioning spike (#1193)

**Throwaway quality evaluation, not production.** Answers one question: does
Stable Audio 3 Medium, conditioned on a source stem, produce a *variation of
that stem* (recognizable source + applied prompt) rather than a fresh,
unrelated track? That is the only thing that justifies adopting it over the
already-shipped feature-conditioned Lyria path (resonate#1192).

### GPU requirement
Stable Audio 3 Medium needs **FlashAttention-2 → an Ampere+ GPU (compute ≥ 8.0):**
L4, A100, RTX 3090/4090. A **free-Colab T4 (Turing 7.5) will NOT work.** Use
Colab Pro (L4/A100) or an external L4/4090 pod.

### What you do
1. Set an Ampere+ GPU runtime (Runtime → Change runtime type → L4/A100).
2. Run each cell top to bottom; paste your Hugging Face token when prompted
   (accept the model license first at huggingface.co/stabilityai/stable-audio-3-medium).
3. Listen to the outputs, score them with the rubric at the bottom, and send
   me `metrics.json` + your scores. Then **delete the runtime** (it's throwaway).


## 1. Confirm the GPU supports FlashAttention-2


In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU — pick a GPU runtime.'
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f'GPU: {name}  compute capability: {cap[0]}.{cap[1]}')
if cap[0] < 8:
    print('\n[!] This GPU is pre-Ampere (compute < 8.0). FlashAttention-2 is',
          'required by Stable Audio 3 Medium and will not run here.',
          '\n    Switch to an L4 or A100 runtime (Colab Pro) or an external L4/4090.')
else:
    print('OK — Ampere+, FlashAttention-2 supported.')


## 2. Install dependencies
flash-attn builds from source and can take several minutes — expected.


In [ ]:
%pip install -q torchaudio soundfile numpy
%pip install -q flash-attn --no-build-isolation
%pip install -q 'git+https://github.com/Stability-AI/stable-audio-3.git'
print('deps installed')


## 3. Hugging Face auth (gated model)
The token is read via getpass and **not stored in the notebook**.


In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token (read scope): ').strip()
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('authenticated')


## 4. Conditioning stem
A short synthetic musical loop (Am chord arpeggio at ~90 BPM) so the notebook
is self-contained. **To test a real Resonate stem instead**, upload a wav/mp3
with the Colab file panel and set `STEM_PATH` to its filename.


In [ ]:
import numpy as np, soundfile as sf
SR = 44100
STEM_PATH = 'source_stem.wav'  # <- change to your uploaded file to use a real stem

def synth_arp(seconds=8.0, bpm=90):
    # A minor arpeggio (A2 C3 E3 A3) as a simple, tonal conditioning source.
    notes = [110.0, 130.81, 164.81, 220.0]
    step = 60.0 / bpm / 2  # eighth notes
    t = np.arange(int(seconds*SR))/SR
    out = np.zeros_like(t)
    for i in range(int(seconds/step)):
        f = notes[i % len(notes)]
        s = int(i*step*SR); e = int((i*step+step)*SR)
        env = np.linspace(1,0,e-s)**1.5
        out[s:e] += 0.5*np.sin(2*np.pi*f*t[s:e])*env
    return (out/np.max(np.abs(out))*0.9).astype(np.float32)

import os
if not os.path.exists(STEM_PATH) or STEM_PATH == 'source_stem.wav':
    sf.write('source_stem.wav', synth_arp(), SR)
    STEM_PATH = 'source_stem.wav'
print('conditioning stem:', STEM_PATH)


## 5. Run the init_noise_level sweep
Low noise = close variation (more source preserved); high = looser. We sweep
to find the band where the source stays recognizable *and* the prompt applies.

> The two `# RECONCILE` lines are the only spots to adjust if the installed
> `stable_audio_3` API differs from the documented snippets (reference-audio
> load + `generate()` return shape).


In [ ]:
import json, time, torch, torchaudio
from stable_audio_3 import StableAudioModel

PROMPT = 'darker halftime variation, keep the groove'  # edit freely
NOISE_LEVELS = [0.3, 0.5, 0.7, 0.9]
DURATION = 30

t0 = time.monotonic()
model = StableAudioModel.from_pretrained('medium', device='cuda')
print(f'model loaded in {time.monotonic()-t0:.1f}s')

init_audio = torchaudio.load(STEM_PATH)  # RECONCILE: documented as (tensor, sr) tuple
metrics = {'gpu': torch.cuda.get_device_name(0), 'prompt': PROMPT,
           'durationSeconds': DURATION, 'runs': []}
outputs = []
for noise in NOISE_LEVELS:
    torch.cuda.reset_peak_memory_stats()
    g0 = time.monotonic()
    audio = model.generate(init_audio=init_audio, init_noise_level=noise,
                           prompt=PROMPT, duration=DURATION, seed=1189)
    gen_s = round(time.monotonic()-g0, 2)
    vram = round(torch.cuda.max_memory_allocated()/1e9, 2)
    tensor, sr = (audio if isinstance(audio, tuple) and len(audio)==2
                  else (audio, 44100))  # RECONCILE: generate() return shape
    path = f'out_noise_{noise:.2f}.wav'
    torchaudio.save(path, tensor.detach().cpu(), sr)
    outputs.append((noise, path))
    metrics['runs'].append({'initNoiseLevel': noise, 'outputFile': path,
                            'generationSeconds': gen_s, 'peakVramGb': vram})
    print(f'noise={noise:.2f} -> {path}  ({gen_s}s, peak {vram} GB)')

json.dump(metrics, open('metrics.json','w'), indent=2)
print('\nwrote metrics.json')


## 6. Listen — source vs. each output


In [ ]:
from IPython.display import Audio, display
import IPython.display as ipd
print('SOURCE stem:'); display(Audio(STEM_PATH))
for noise, path in outputs:
    print(f'init_noise_level = {noise}:'); display(Audio(path))


## 7. Score it (the human verdict)

For each output, rate 1–5:
1. **Source identity** — can you still hear the original stem? (< 3 ⇒ it's not
   a remix of *this* audio — the whole point fails)
2. **Prompt adherence** — did the requested change happen?
3. **Musical quality** — coherent, not garbled?
4. **Usable as a draft** — would a creator accept it as a starting point?

**Decision:** find the `init_noise_level` band where identity AND prompt are
both ≥ 3. If none exists → audio conditioning doesn't deliver for our use case
→ **reject**, stay on feature-conditioned Lyria (#1192) + stem-mix renders (#1189).

Send me `metrics.json` (latency/VRAM) + your per-clip scores and I'll write up
`docs/rfc/stable-audio-3-spike-findings.md` and the #1182 slices 4–5 decision.
Then **delete this runtime** — it's throwaway.
